# Matrix Norms — Exercises

8 exercises covering matrix norm computation, properties, condition numbers, perturbation theory, nuclear norm optimization, and ML applications.

| Format | Description |
| --- | --- |
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding (`# YOUR CODE HERE`) |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
| --- | --- | --- |
| ★ | 1–3 | Core mechanics: computing norms, verifying inequalities |
| ★★ | 4–6 | Deeper theory: perturbation, duality, optimization |
| ★★★ | 7–8 | AI applications: spectral normalization, implicit regularization |

### Topic Map

| Topic | Exercises |
| --- | --- |
| Frobenius norm properties | 1, 2 |
| Condition number and stability | 3 |
| Weyl's inequality | 4 |
| Nuclear norm and duality | 5 |
| Low-rank approximation (Eckart-Young) | 6 |
| Spectral normalization (GAN / Lipschitz) | 7 |
| Implicit regularization in matrix factorization | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [9, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '='*len(title))
    print(title)
    print('='*len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f'{"PASS" if ok else "FAIL"} — {name}')
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f'{"PASS" if cond else "FAIL"} — {name}')
    return cond

def svt(A, tau):
    """Singular value thresholding: proximal operator of tau * ||.||_*"""
    U, s, Vt = la.svd(A, full_matrices=False)
    return U @ np.diag(np.maximum(s - tau, 0)) @ Vt

def stable_rank(A):
    """Stable rank: ||A||_F^2 / ||A||_2^2"""
    s = la.svdvals(A)
    return np.sum(s**2) / s[0]**2

print('Setup complete.')

---

## Exercise 1 ★ — Norm Computation and Verification

Let $A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & -1 \\ 1 & 0 & 4 \end{pmatrix}$.

**(a)** Compute $\|A\|_F$ (Frobenius) using the elementwise formula and verify it equals $\|\boldsymbol{\sigma}\|_2$.

**(b)** Compute $\|A\|_1$ (max absolute column sum) and $\|A\|_\infty$ (max absolute row sum). Verify $\|A^\top\|_1 = \|A\|_\infty$.

**(c)** Compute $\|A\|_2 = \sigma_1(A)$ and verify $\|A\|_2 \leq \|A\|_F \leq \sqrt{r}\|A\|_2$.

**(d)** Compute $\|A\|_*$ and verify $\|A\|_F \leq \|A\|_* \leq \sqrt{r}\|A\|_F$.

In [ ]:
# Exercise 1: Your Solution

A = np.array([[3., 1., 0.],
              [0., 2., -1.],
              [1., 0., 4.]])

# (a) Frobenius norm
fro_elementwise = None   # YOUR CODE HERE
fro_svd         = None   # YOUR CODE HERE (use la.svdvals)

# (b) 1-norm and inf-norm
norm1   = None   # YOUR CODE HERE
norminf = None   # YOUR CODE HERE

# (c) Spectral norm
spec = None   # YOUR CODE HERE
r    = None   # YOUR CODE HERE (rank of A)

# (d) Nuclear norm
nuc = None   # YOUR CODE HERE

print(f'(a) ||A||_F elementwise: {fro_elementwise}')
print(f'(a) ||A||_F via SVD:     {fro_svd}')
print(f'(b) ||A||_1: {norm1},  ||A||_inf: {norminf}')
print(f'(c) ||A||_2: {spec},  r={r}')
print(f'(d) ||A||_*: {nuc}')

In [ ]:
# Exercise 1: Solution

A = np.array([[3., 1., 0.],
              [0., 2., -1.],
              [1., 0., 4.]])

# (a) Frobenius
fro_elementwise = np.sqrt(np.sum(A**2))
sigma = la.svdvals(A)
fro_svd = la.norm(sigma)  # ell-2 of singular values

# (b) 1-norm and inf-norm
norm1   = la.norm(A, 1)
norminf = la.norm(A, np.inf)

# (c) Spectral norm
spec = la.norm(A, 2)
r    = np.linalg.matrix_rank(A)

# (d) Nuclear norm
nuc = la.norm(A, 'nuc')

header('Exercise 1: Norm Computation')
check_close('(a) Frobenius: elementwise == SVD formula', fro_elementwise, fro_svd)
check_true ('(b) 1-norm: max column sum', np.isclose(norm1, max(np.sum(np.abs(A), axis=0))))
check_true ('(b) inf-norm = ||A^T||_1', np.isclose(norminf, la.norm(A.T, 1)))
check_true ('(c) ||A||_2 <= ||A||_F', spec <= fro_elementwise + 1e-10)
check_true ('(c) ||A||_F <= sqrt(r)*||A||_2', fro_elementwise <= np.sqrt(r)*spec + 1e-10)
check_true ('(d) ||A||_F <= ||A||_*', fro_elementwise <= nuc + 1e-10)
check_true ('(d) ||A||_* <= sqrt(r)*||A||_F', nuc <= np.sqrt(r)*fro_elementwise + 1e-10)

print(f'\nSingular values: {np.round(sigma, 4)}')
print(f'||A||_F={fro_elementwise:.4f}, ||A||_2={spec:.4f}, ||A||_*={nuc:.4f}')
print('\nTakeaway: spectral <= Frobenius <= nuclear; all bounded by sqrt(rank) multiples')

---

## Exercise 2 ★ — Frobenius Norm Properties

**(a)** Verify submultiplicativity $\|AB\|_F \leq \|A\|_F \|B\|_F$ for 100 random pairs. Report the tightest ratio $\|AB\|_F / (\|A\|_F \|B\|_F)$.

**(b)** For a rank-1 matrix $A = \mathbf{u}\mathbf{v}^\top$, show analytically and verify numerically that $\|A\|_F = \|\mathbf{u}\|_2 \|\mathbf{v}\|_2 = \|A\|_2$.

**(c)** Compute the gradient of $f(W) = \|W\|_F^2$ at a random $W \in \mathbb{R}^{4 \times 5}$ and verify it equals $2W$ using finite differences.

**(d)** Show numerically that the Frobenius norm is NOT induced: find a matrix $A$ such that $\|A\|_F > \max_{\|x\|_2=1} \|Ax\|_2$.

In [ ]:
# Exercise 2: Your Solution

np.random.seed(42)

# (a) Submultiplicativity
ratios = []
for _ in range(100):
    A = np.random.randn(4, 5)
    B = np.random.randn(5, 3)
    ratio = None   # YOUR CODE HERE: ||AB||_F / (||A||_F * ||B||_F)
    ratios.append(ratio)

# (b) Rank-1 matrix
u = np.array([1., 2., 3.])
v = np.array([4., 5.])
A_rank1 = np.outer(u, v)
fro_rank1  = None   # YOUR CODE HERE
spec_rank1 = None   # YOUR CODE HERE
expected   = None   # YOUR CODE HERE: ||u|| * ||v||

# (c) Gradient of ||W||_F^2
W = np.random.randn(4, 5)
exact_grad = None   # YOUR CODE HERE

# (d) Not induced
A2 = np.eye(3)
fro_I = None    # YOUR CODE HERE: ||I||_F
spec_I = None   # YOUR CODE HERE: ||I||_2

print('(a) Min ratio (should be < 1):', min(r for r in ratios if r is not None))
print('(b) ||A_rank1||_F:', fro_rank1, '  ||u||*||v||:', expected)

In [ ]:
# Exercise 2: Solution

np.random.seed(42)

# (a) Submultiplicativity
ratios = []
for _ in range(100):
    A = np.random.randn(4, 5)
    B = np.random.randn(5, 3)
    ratio = la.norm(A @ B, 'fro') / (la.norm(A, 'fro') * la.norm(B, 'fro'))
    ratios.append(ratio)

# (b) Rank-1
u = np.array([1., 2., 3.])
v = np.array([4., 5.])
A_rank1 = np.outer(u, v)
fro_rank1  = la.norm(A_rank1, 'fro')
spec_rank1 = la.norm(A_rank1, 2)
expected   = la.norm(u) * la.norm(v)

# (c) Gradient via finite differences
W = np.random.randn(4, 5)
exact_grad = 2 * W
eps = 1e-6
fd_grad = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        Wp = W.copy(); Wp[i,j] += eps
        Wm = W.copy(); Wm[i,j] -= eps
        fd_grad[i,j] = (la.norm(Wp,'fro')**2 - la.norm(Wm,'fro')**2)/(2*eps)

# (d) Not induced: ||I_n||_F = sqrt(n) != 1
A2 = np.eye(4)
fro_I  = la.norm(A2, 'fro')   # = sqrt(4) = 2
spec_I = la.norm(A2, 2)        # = 1

header('Exercise 2: Frobenius Norm Properties')
check_true('(a) all ratios <= 1 (submultiplicativity)', max(ratios) <= 1.0 + 1e-10)
print(f'    tightest ratio: {max(ratios):.4f}')
check_close('(b) ||rank-1||_F = ||u||*||v||', fro_rank1, expected)
check_close('(b) ||rank-1||_F = ||rank-1||_2', fro_rank1, spec_rank1)
check_close('(c) gradient of ||W||_F^2 is 2W', exact_grad, fd_grad, tol=1e-4)
check_true('(d) ||I_4||_F = 2 != 1 = ||I_4||_2 (NOT induced)', not np.isclose(fro_I, spec_I))
print(f'    ||I_4||_F={fro_I:.2f}, ||I_4||_2={spec_I:.2f}  -> Frobenius is not induced')

print('\nTakeaway: Frobenius satisfies ||I||=sqrt(n)!=1, so cannot be induced by any vector norm')

---

## Exercise 3 ★ — Condition Number and Numerical Stability

**(a)** Construct matrices with $\kappa_2 \approx 10$, $\kappa_2 \approx 10^4$, and $\kappa_2 \approx 10^8$.

**(b)** For each, solve $A\mathbf{x} = \mathbf{b}$ with $\mathbf{b} = A\mathbf{1}$ (exact solution $\mathbf{1}$). Add noise $\delta\mathbf{b} = 10^{-8}\cdot\text{rand}$ and measure the relative error in $\hat{\mathbf{x}}$.

**(c)** Verify the bound: relative forward error $\leq \kappa(A) \cdot$ relative backward error.

**(d)** Apply Tikhonov regularization with $\lambda = 10^{-4}$ to the ill-conditioned matrix. Report the new $\kappa$ and solution accuracy.

In [ ]:
# Exercise 3: Your Solution

np.random.seed(42)
n = 5

def make_cond(n, kappa):
    Q, _ = la.qr(np.random.randn(n, n))
    sigmas = np.logspace(0, np.log10(kappa), n)
    return Q @ np.diag(sigmas) @ Q.T

# (a) Build matrices
A_10   = make_cond(n, 10)
A_1e4  = make_cond(n, 1e4)
A_1e8  = make_cond(n, 1e8)

x_true = np.ones(n)
eps_noise = 1e-8

# (b) Solve with noise
for A, name in [(A_10, 'kappa=10'), (A_1e4, 'kappa=1e4'), (A_1e8, 'kappa=1e8')]:
    kappa  = None   # YOUR CODE HERE
    b      = A @ x_true
    b_noisy = b + eps_noise * np.random.randn(n)
    x_hat  = None   # YOUR CODE HERE: solve A x = b_noisy
    rel_err  = None  # YOUR CODE HERE
    print(f'{name}: kappa={kappa}, rel_err={rel_err}')

In [ ]:
# Exercise 3: Solution

np.random.seed(42)
n = 5

def make_cond(n, kappa):
    Q, _ = la.qr(np.random.randn(n, n))
    sigmas = np.logspace(0, np.log10(kappa), n)
    return Q @ np.diag(sigmas) @ Q.T

A_10  = make_cond(n, 10)
A_1e4 = make_cond(n, 1e4)
A_1e8 = make_cond(n, 1e8)

x_true = np.ones(n)
eps_noise = 1e-8

header('Exercise 3: Condition Number')
for A, kappa_target in [(A_10, 10), (A_1e4, 1e4), (A_1e8, 1e8)]:
    b       = A @ x_true
    b_noisy = b + eps_noise * np.random.randn(n)
    x_hat   = la.solve(A, b_noisy)
    kappa   = la.cond(A)
    rel_err  = la.norm(x_hat - x_true) / la.norm(x_true)
    rel_back = la.norm(b_noisy - b) / la.norm(b)
    bound    = kappa * rel_back
    ok = rel_err <= bound + 1e-10
    print(f'  kappa~{kappa_target:.0e}: rel_err={rel_err:.2e}, bound={bound:.2e}, PASS={ok}')

# (d) Tikhonov regularization
A = A_1e8
b = A @ x_true + 1e-6 * np.random.randn(n)
lam = 1e-4

# Regularized solve
x_reg = la.solve(A + lam * np.eye(n), b)
kappa_reg = la.cond(A + lam * np.eye(n))
err_reg   = la.norm(x_reg - x_true) / la.norm(x_true)

print(f'\n(d) Tikhonov (lambda={lam}):')
print(f'    kappa(A)      = {la.cond(A):.2e}')
print(f'    kappa(A+lI)   = {kappa_reg:.2e}  (improved!)')
print(f'    rel_err       = {err_reg:.4f}  (bias from regularization)')

print('\nTakeaway: condition number determines precision loss; Tikhonov trades bias for stability')

---

## Exercise 4 ★★ — Weyl's Inequality and Perturbation

**(a)** Verify Weyl's inequality: for $A, E \in \mathbb{R}^{6 \times 5}$, verify $|\sigma_i(A+E) - \sigma_i(A)| \leq \|E\|_2$ for all $i$.

**(b)** Construct $A$ and $E$ such that equality is achieved for $i=1$. Hint: make $E = \epsilon \mathbf{u}_1\mathbf{v}_1^\top$.

**(c)** For a non-normal matrix, compare eigenvalue sensitivity to singular value sensitivity over 200 random perturbations with $\|E\|_2 = 0.1$. Which is more stable?

**(d)** Show the Mirsky extension: $\|\boldsymbol{\sigma}(A+E) - \boldsymbol{\sigma}(A)\|_2 \leq \|E\|_F$.

In [ ]:
# Exercise 4: Your Solution

import numpy as np
import numpy.linalg as la
np.random.seed(42)

# (a) Verify Weyl
A = np.random.randn(6, 5)
E = np.random.randn(6, 5) * 0.5

sigma_A  = la.svdvals(A)
sigma_AE = None  # YOUR CODE HERE
norm_E2  = None  # YOUR CODE HERE

diffs = None  # YOUR CODE HERE: |sigma_i(A+E) - sigma_i(A)| for each i
print('Weyl check (each diff <= ||E||_2):')
for i, d in enumerate(diffs or []):
    print(f'  i={i}: diff={d:.4f}, bound={norm_E2:.4f}')

# (b) Tight example -- YOUR CODE HERE

# (c) Eigenvalue vs singular value sensitivity -- YOUR CODE HERE

# (d) Mirsky -- YOUR CODE HERE
print('Sketch your solutions above')

In [ ]:
# Exercise 4: Solution

np.random.seed(42)

# (a) Weyl's inequality
A = np.random.randn(6, 5)
E = np.random.randn(6, 5) * 0.5

sigma_A  = la.svdvals(A)
sigma_AE = la.svdvals(A + E)
norm_E2  = la.norm(E, 2)
norm_E_F = la.norm(E, 'fro')
diffs = np.abs(sigma_AE - sigma_A)

header = lambda t: print('\n'+'='*len(t)+'\n'+t+'\n'+'='*len(t))
header('Exercise 4: Weyl\'s Inequality')

print('(a) Weyl check:')
for i, d in enumerate(diffs):
    ok = d <= norm_E2 + 1e-10
    print(f'  i={i}: |Δσ|={d:.4f} <= ||E||_2={norm_E2:.4f}  PASS={ok}')
print(f'PASS - Weyl holds for all i: {np.all(diffs <= norm_E2 + 1e-10)}')

# (b) Tight example: E = eps * u1 v1^T
U, s, Vt = la.svd(A)
eps = 0.3
E_tight = eps * np.outer(U[:, 0], Vt[0])  # rank-1 perturbation
sigma_tight = la.svdvals(A + E_tight)
diff_i1 = abs(sigma_tight[0] - s[0])
norm_E_tight = la.norm(E_tight, 2)
print(f'\n(b) Tight example: |Δσ_1|={diff_i1:.4f}, ||E||_2={norm_E_tight:.4f}')
print(f'    Ratio (should be ~1): {diff_i1/norm_E_tight:.4f}')

# (c) Eigenvalue vs singular value sensitivity
n = 5
A_nn = np.triu(np.random.randn(n, n))  # non-normal upper triangular
eps_val = 0.1

sv_diffs, ev_diffs = [], []
for _ in range(200):
    E_rand = eps_val * np.random.randn(n, n)
    E_rand = E_rand / la.norm(E_rand, 2)  # ||E||_2 = 1, scaled by eps
    E_rand *= eps_val
    sv_diffs.append(np.max(np.abs(la.svdvals(A_nn + E_rand) - la.svdvals(A_nn))))
    ev_diffs.append(np.max(np.abs(np.abs(la.eigvals(A_nn + E_rand)) - np.abs(la.eigvals(A_nn)))))

print(f'\n(c) Mean |Δσ| (singular values): {np.mean(sv_diffs):.4f}')
print(f'    Mean |Δλ| (eigenvalues):       {np.mean(ev_diffs):.4f}')
print(f'    Eigenvalues {np.mean(ev_diffs)/np.mean(sv_diffs):.1f}x more sensitive than singular values')

# (d) Mirsky extension
mirsky_lhs = la.norm(sigma_AE - sigma_A)  # ||Δσ||_2
print(f'\n(d) Mirsky: ||Δσ||_2={mirsky_lhs:.4f} <= ||E||_F={norm_E_F:.4f}')
print(f'    PASS: {mirsky_lhs <= norm_E_F + 1e-10}')

print('\nTakeaway: Weyl makes singular values numerically stable; eigenvalues of non-normal are not')

---

## Exercise 5 ★★ — Nuclear Norm and Duality

**(a)** Verify the duality $\|A\|_* = \max_{\|B\|_2 \leq 1}\operatorname{tr}(A^\top B)$ numerically for a random $5 \times 4$ matrix. Show the maximizer is $B = UV^\top$.

**(b)** Implement SVT (singular value thresholding) and verify it is the proximal operator:
$$\operatorname{prox}_{\tau\|\cdot\|_*}(A) = \arg\min_X \frac{1}{2}\|X-A\|_F^2 + \tau\|X\|_*$$

**(c)** Apply 50 iterations of proximal gradient to minimize $\|A\|_*$ subject to $P_\Omega(X) = P_\Omega(A)$ (matrix completion with 50% observations). Report the nuclear norm and rank of the recovered matrix.

**(d)** Show that the factorization formula $\|A\|_* = \min_{A=UV^\top}\frac{1}{2}(\|U\|_F^2 + \|V\|_F^2)$ holds, with minimum achieved at the SVD factorization.

In [ ]:
# Exercise 5: Your Solution

np.random.seed(7)
A = np.random.randn(5, 4)

# (a) Duality: ||A||_* = max_{||B||_2<=1} tr(A^T B)
nuc_direct = la.norm(A, 'nuc')
U, s, Vt = la.svd(A, full_matrices=False)
B_opt = None   # YOUR CODE HERE: optimal B from SVD
trace_opt = None  # YOUR CODE HERE: tr(A^T B_opt)

# (b) SVT proximal operator
def svt(A, tau):
    # YOUR CODE HERE
    pass

# (c) Matrix completion -- YOUR CODE HERE (sketch)

# (d) Factorization formula
# ||A||_* = min_{A=UV^T} (1/2)(||U||_F^2 + ||V||_F^2)
# At optimal factorization U=U0*Sigma^{1/2}, V=V0*Sigma^{1/2}
U0, s0, Vt0 = la.svd(A, full_matrices=False)
U_opt = U0 * np.sqrt(s0)
V_opt = Vt0.T * np.sqrt(s0)
factor_val = None   # YOUR CODE HERE: (1/2)(||U_opt||_F^2 + ||V_opt||_F^2)

print(f'(a) ||A||_* = {nuc_direct:.4f},  tr(A^T B_opt) = {trace_opt}')
print(f'(d) Factorization value = {factor_val}')

In [ ]:
# Exercise 5: Solution

np.random.seed(7)
A = np.random.randn(5, 4)

header_fn = lambda t: print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
header_fn('Exercise 5: Nuclear Norm and Duality')

# (a) Duality
nuc_direct = la.norm(A, 'nuc')
U, s, Vt = la.svd(A, full_matrices=False)
B_opt = U @ Vt  # optimal B
trace_opt = np.trace(A.T @ B_opt)
norm_B_opt = la.norm(B_opt, 2)

print(f'(a) ||A||_* = {nuc_direct:.6f}')
print(f'    tr(A^T B_opt) = {trace_opt:.6f}')
print(f'    ||B_opt||_2 = {norm_B_opt:.6f}  (should be 1)')
print(f'    PASS - duality: {np.isclose(nuc_direct, trace_opt)}')

# (b) SVT
def svt(A, tau):
    U_, s_, Vt_ = la.svd(A, full_matrices=False)
    return U_ @ np.diag(np.maximum(s_ - tau, 0)) @ Vt_

tau = 1.0
X_svt = svt(A, tau)
print(f'\n(b) SVT(A, tau={tau}):')
print(f'    singular values: {np.round(la.svdvals(X_svt), 4)}')
print(f'    PASS - svt reduces nuc norm: {la.norm(X_svt, "nuc") < la.norm(A, "nuc")}')

# (c) Matrix completion via proximal gradient
m, n_c = 8, 6
r_true = 2
M_true = np.random.randn(m, r_true) @ np.random.randn(r_true, n_c)
Omega = np.random.rand(m, n_c) < 0.5
M_obs = M_true * Omega

lam, step = 0.1, 0.5
X = np.zeros((m, n_c))
for t in range(50):
    grad = (X - M_obs) * Omega
    X = svt(X - step * grad, step * lam)

print(f'\n(c) Matrix completion (rank-{r_true}, 50% obs):')
print(f'    Recovery error: {la.norm(X - M_true, "fro"):.4f}')
print(f'    Rank of X:      {np.linalg.matrix_rank(X, tol=1e-4)}')

# (d) Factorization formula
U0, s0, Vt0 = la.svd(A, full_matrices=False)
U_opt = U0 * np.sqrt(s0)
V_opt = Vt0.T * np.sqrt(s0)
factor_val = 0.5 * (la.norm(U_opt,'fro')**2 + la.norm(V_opt,'fro')**2)
print(f'\n(d) Factorization: (1/2)(||U||_F^2+||V||_F^2) = {factor_val:.6f}')
print(f'    ||A||_* = {nuc_direct:.6f}')
print(f'    PASS: {np.isclose(factor_val, nuc_direct)}')

print('\nTakeaway: nuclear norm = convex proxy for rank; SVT is its proximal operator')

---

## Exercise 6 ★★ — Eckart-Young: Low-Rank Approximation

**(a)** For a $10 \times 8$ weight matrix $W$, compute best rank-$k$ approximations $W_k$ for $k = 1, 2, 3, 5, 8$ and verify Eckart-Young:
$$\|W - W_k\|_F = \sqrt{\sigma_{k+1}^2 + \cdots + \sigma_r^2}, \quad \|W - W_k\|_2 = \sigma_{k+1}$$

**(b)** Simulate LoRA: pretend $W = W_0 + \Delta W$ where $\Delta W$ is a rank-3 perturbation. For $r \in \{1, 2, 3, 5, 8\}$, find the best rank-$r$ approximation to $\Delta W$ and report the approximation error $\|\Delta W - B_rA_r\|_F$.

**(c)** Compute the compression ratio for each rank $r$ (parameters in rank-$r$ approximation vs parameters in $\Delta W$).

**(d)** Plot the approximation error vs compression ratio for $r \in \{1,\ldots,\min(m,n)\}$.

In [ ]:
# Exercise 6: Your Solution

np.random.seed(3)
m, n_w = 10, 8
W = np.random.randn(m, n_w)
U, s, Vt = la.svd(W)

# (a) Best rank-k approximations
print('k  ||W-W_k||_F  sigma_{k+1}  ||W-W_k||_2')
for k in [1, 2, 3, 5, 8]:
    W_k = None        # YOUR CODE HERE: rank-k truncated SVD
    err_F  = None     # YOUR CODE HERE
    err_2  = None     # YOUR CODE HERE
    expected_F = None # YOUR CODE HERE: sqrt(sum sigma[k:]^2)
    print(f'{k}  {err_F:.4f}  {expected_F:.4f}  {err_2:.4f}')

# (b) LoRA simulation -- YOUR CODE HERE

# (c) Compression ratios -- YOUR CODE HERE

print('(a)-(c) solution sketched above; see solution cell for reference')

In [ ]:
# Exercise 6: Solution

np.random.seed(3)
m, n_w = 10, 8
W = np.random.randn(m, n_w)
U, s, Vt = la.svd(W)

def best_rank_k(A, k):
    """Best rank-k approximation via truncated SVD."""
    U_, s_, Vt_ = la.svd(A, full_matrices=False)
    return U_[:, :k] @ np.diag(s_[:k]) @ Vt_[:k, :]

header_fn = lambda t: print('\n'+'='*len(t)+'\n'+t+'\n'+'='*len(t))
header_fn('Exercise 6: Eckart-Young')

print('(a) Eckart-Young verification:')
print(f'    {"k":3s}  {"||W-Wk||_F":>12s}  {"predicted_F":>12s}  {"||W-Wk||_2":>12s}  {"sigma_{k+1}":>12s}  PASS')
all_pass = True
for k in [1, 2, 3, 5, 7]:
    W_k     = best_rank_k(W, k)
    err_F   = la.norm(W - W_k, 'fro')
    err_2   = la.norm(W - W_k, 2)
    pred_F  = np.sqrt(np.sum(s[k:]**2))
    pred_2  = s[k] if k < len(s) else 0.0
    ok = np.isclose(err_F, pred_F) and np.isclose(err_2, pred_2)
    all_pass = all_pass and ok
    print(f'    {k:3d}  {err_F:12.6f}  {pred_F:12.6f}  {err_2:12.6f}  {pred_2:12.6f}  {ok}')
print(f'    PASS all - Eckart-Young: {all_pass}')

# (b) LoRA simulation
r_true = 3
U_dt = np.random.randn(m, r_true)
V_dt = np.random.randn(n_w, r_true)
DeltaW = 0.5 * (U_dt @ V_dt.T)

print(f'\n(b) LoRA: DeltaW is rank-{r_true}, ||DeltaW||_F={la.norm(DeltaW,"fro"):.4f}')
print(f'    {"r":3s}  {"params":>8s}  {"||DeltaW-BrAr||_F":>20s}  {"compression":>12s}')
for r in [1, 2, 3, 5, 8]:
    approx = best_rank_k(DeltaW, r)
    err = la.norm(DeltaW - approx, 'fro')
    params = r * (m + n_w)
    total  = m * n_w
    comp   = total / params
    print(f'    {r:3d}  {params:8d}  {err:20.6f}  {comp:10.1f}x')

print('\nTakeaway: Eckart-Young guarantees truncated SVD is optimal; LoRA captures this optimally')

---

## Exercise 7 ★★★ — Spectral Normalization for Neural Networks

Implement and study spectral normalization as used in GAN training (Miyato 2018).

**(a)** Implement power iteration to estimate $\sigma_1(W)$ for a weight matrix. Compare to the SVD value. Track convergence over 30 steps.

**(b)** For a 3-layer MLP with random weights, compute the Lipschitz bound $\prod_l \|W_l\|_2$ before and after spectral normalization.

**(c)** Train a linear model on toy data with and without spectral normalization. Track the spectral norm of the weight matrix during training. Show that SN keeps $\|W\|_2 \leq 1$ throughout.

**(d)** Demonstrate that larger learning rates are stable with SN but cause divergence without.

In [ ]:
# Exercise 7: Your Solution

import numpy as np
import numpy.linalg as la
np.random.seed(42)

# (a) Power iteration
W = np.random.randn(64, 32)
true_sigma1 = la.svdvals(W)[0]

def power_iter_sigma1(W, n_iter=30):
    # YOUR CODE HERE
    # Return list of estimates at each iteration
    estimates = []
    return estimates

estimates = power_iter_sigma1(W)
print(f'True sigma_1: {true_sigma1:.6f}')
print(f'Power iter estimate after 30 steps: {estimates[-1] if estimates else None}')

# (b) Multi-layer Lipschitz bound
layer_sizes = [(32, 64), (64, 32), (32, 16)]
weights = [np.random.randn(out, inp) for out, inp in layer_sizes]

lip_before = None  # YOUR CODE HERE: product of ||W_l||_2
# Apply SN: divide each W by its sigma_1
weights_sn = [W / la.svdvals(W)[0] for W in weights]  # reference
lip_after  = None  # YOUR CODE HERE

print(f'Lipschitz before SN: {lip_before}')
print(f'Lipschitz after SN:  {lip_after}')

In [ ]:
# Exercise 7: Solution

np.random.seed(42)

def header_fn(t):
    print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))

# (a) Power iteration
W = np.random.randn(64, 32)
true_sigma1 = la.svdvals(W)[0]

def power_iter_sigma1(W, n_iter=30, seed=0):
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(W.shape[1])
    v /= la.norm(v)
    estimates = []
    for _ in range(n_iter):
        u = W @ v;  u /= la.norm(u)
        v = W.T @ u;  sigma = la.norm(v);  v /= sigma
        estimates.append(sigma)
    return estimates

estimates = power_iter_sigma1(W)
sigma2 = la.svdvals(W)[1]
ratio = sigma2 / true_sigma1

header_fn('Exercise 7: Spectral Normalization')
print(f'(a) True sigma_1 = {true_sigma1:.6f}')
for k in [1, 5, 10, 20, 30]:
    err = abs(estimates[k-1] - true_sigma1)
    print(f'    After {k:2d} iters: sigma_1={estimates[k-1]:.6f}, err={err:.2e}')
print(f'    Conv. rate (sigma_2/sigma_1)^2 = {ratio**2:.4f}')

# (b) Multi-layer Lipschitz
layer_sizes = [(32, 64), (64, 32), (32, 16)]
weights = [np.random.randn(out, inp) for out, inp in layer_sizes]

lip_before = 1.0
for W_l in weights:
    lip_before *= la.svdvals(W_l)[0]

weights_sn = [W_l / la.svdvals(W_l)[0] for W_l in weights]
lip_after = 1.0
for W_l in weights_sn:
    lip_after *= la.svdvals(W_l)[0]

print(f'\n(b) Lipschitz bound before SN: {lip_before:.4f}')
print(f'    Lipschitz bound after SN:  {lip_after:.6f}  (exactly 1)')
print(f'    PASS: SN enforces Lipschitz=1: {np.isclose(lip_after, 1.0)}')

# (c) Training simulation: binary classification toy problem
np.random.seed(5)
n_data, d = 50, 8
X = np.random.randn(n_data, d)
y = (X @ np.random.randn(d) > 0).astype(float)  # binary labels

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -10, 10)))
def bce_loss(W, X, y): return -np.mean(y * np.log(sigmoid(X @ W.T[0]) + 1e-9) + (1-y)*np.log(1-sigmoid(X @ W.T[0]) + 1e-9))

W_plain = np.random.randn(1, d) * 0.1
W_sn    = W_plain.copy()
lr = 0.5

spec_plain, spec_sn = [], []
for t in range(100):
    # Gradient for plain
    err = sigmoid(X @ W_plain.T[0]) - y
    G_plain = (err[:, None] * X).mean(0, keepdims=True)
    W_plain = W_plain - lr * G_plain
    spec_plain.append(la.norm(W_plain, 2))

    # Gradient for SN (normalize after update)
    err_sn = sigmoid(X @ W_sn.T[0]) - y
    G_sn = (err_sn[:, None] * X).mean(0, keepdims=True)
    W_sn = W_sn - lr * G_sn
    sigma_sn = la.norm(W_sn, 2)
    if sigma_sn > 1.0:
        W_sn = W_sn / sigma_sn  # spectral normalization
    spec_sn.append(la.norm(W_sn, 2))

print(f'\n(c) After 100 steps (lr={lr}):')
print(f'    Plain: final ||W||_2 = {spec_plain[-1]:.4f}')
print(f'    SN:    final ||W||_2 = {spec_sn[-1]:.4f}  (bounded by 1)')
print(f'    PASS - SN keeps ||W||_2 <= 1: {max(spec_sn) <= 1.0 + 1e-10}')

print('\nTakeaway: spectral normalization enforces Lipschitz-1 layers; product bounds network Lipschitz')

---

## Exercise 8 ★★★ — Implicit Regularization in Matrix Factorization

Gradient flow on $\min_{B,A}\|BA - M\|_F^2$ implicitly minimizes the nuclear norm (Gunasekar et al., 2017). This exercise demonstrates this phenomenon.

**(a)** Minimize $\|BA - M\|_F^2$ via gradient descent (small step size) for a $6\times 6$ matrix $M$ with $B \in \mathbb{R}^{6\times 4}$, $A \in \mathbb{R}^{4\times 6}$. Initialize $B, A$ close to zero. Track $\|BA\|_*$ over iterations.

**(b)** Compare to the minimum nuclear norm solution: use nuclear norm minimization via SVT to find $\min\{\|X\|_* : X = M\}$ and compare $\|BA_{\text{converged}}\|_*$ with $\|M\|_*$.

**(c)** Now let $M$ be rank-2. Show that gradient flow with $r=4$ converges to a rank-2 solution even though the factorization supports rank-4.

**(d)** Demonstrate that smaller initialization (smaller $\|B_0\|_F, \|A_0\|_F$) leads to smaller $\|BA_{\text{converged}}\|_*$. Plot nuclear norm of converged solution vs initialization scale.

In [ ]:
# Exercise 8: Your Solution

np.random.seed(0)

n = 6;  r = 4
M = np.random.randn(n, n)  # full-rank target

# (a) Gradient descent on ||BA - M||_F^2
scale_init = 0.01  # small initialization
B = np.random.randn(n, r) * scale_init
A = np.random.randn(r, n) * scale_init
lr = 1e-3
n_steps = 500

nuc_history = []
for t in range(n_steps):
    # Compute gradients
    # YOUR CODE HERE
    pass

print(f'Initial  ||BA||_* = {la.norm(B @ A, "nuc"):.4f}')
print(f'Converged ||BA||_* = {la.norm(B @ A, "nuc"):.4f}')
print(f'||M||_* = {la.norm(M, "nuc"):.4f}')

In [ ]:
# Exercise 8: Solution

np.random.seed(0)

def header_fn(t):
    print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))

n = 6;  r_fact = 4
M = np.random.randn(n, n)  # full-rank target

# (a) Gradient descent on ||BA - M||_F^2
def train_factored(M, r, lr=1e-3, n_steps=1000, scale=0.01):
    n = M.shape[0]
    B = np.random.randn(n, r) * scale
    A = np.random.randn(r, n) * scale
    nuc_hist = []
    for t in range(n_steps):
        Res = B @ A - M  # residual
        dB  = 2 * Res @ A.T
        dA  = 2 * B.T @ Res
        B = B - lr * dB
        A = A - lr * dA
        if t % 50 == 0:
            nuc_hist.append(la.norm(B @ A, 'nuc'))
    return B, A, nuc_hist

header_fn('Exercise 8: Implicit Regularization')

B, A, nuc_hist = train_factored(M, r_fact)
BA = B @ A
print(f'(a) After 1000 steps:')
print(f'    ||BA - M||_F = {la.norm(BA - M, "fro"):.6f}  (should be near 0 if converged)')
print(f'    ||BA||_* = {la.norm(BA, "nuc"):.4f}')
print(f'    ||M||_*  = {la.norm(M, "nuc"):.4f}')
print(f'    Nuclear norm history (every 50 steps): {[f"{v:.2f}" for v in nuc_hist[:5]]}...')

# (b) Compare to explicit nuclear norm minimization
print(f'\n(b) True nuclear norm of M = {la.norm(M, "nuc"):.4f}')
print(f'    BA converged to ||BA||_* = {la.norm(BA, "nuc"):.4f}')
print(f'    (Should be close to ||M||_* as factored GD ~= nuclear norm min)')

# (c) Rank-2 target: factored GD with r=4 should find rank-2 solution
np.random.seed(1)
U2 = np.random.randn(n, 2)
V2 = np.random.randn(n, 2)
M_rank2 = U2 @ V2.T  # rank-2

B2, A2, _ = train_factored(M_rank2, r=4, lr=1e-3, n_steps=2000, scale=0.01)
BA2 = B2 @ A2
svs_BA2 = la.svdvals(BA2)
eff_rank = np.sum(svs_BA2 > 1e-4)

print(f'\n(c) Rank-2 target, r=4 factorization:')
print(f'    Singular values of BA: {np.round(svs_BA2, 4)}')
print(f'    Effective rank (> 1e-4): {eff_rank}  (should be ~2)')
print(f'    ||BA - M_rank2||_F = {la.norm(BA2 - M_rank2, "fro"):.6f}')

# (d) Nuclear norm vs initialization scale
scales = [0.001, 0.01, 0.05, 0.1, 0.5]
nuc_converged = []
for scale in scales:
    np.random.seed(0)
    B_, A_, _ = train_factored(M, r_fact, lr=5e-4, n_steps=2000, scale=scale)
    nuc_converged.append(la.norm(B_ @ A_, 'nuc'))

print(f'\n(d) Nuclear norm vs initialization scale:')
for sc, nuc in zip(scales, nuc_converged):
    print(f'    scale={sc:.3f}: ||BA||_* = {nuc:.4f}')
print('Smaller init -> smaller nuclear norm (implicit bias toward low nuclear norm)')

print('\nTakeaway: gradient descent on factored matrices implicitly minimizes nuclear norm')

---

## What to Review After Finishing

- [ ] Exercise 1: All five norms computed correctly; inequalities verified
- [ ] Exercise 2: Frobenius submultiplicativity; gradient formula $\nabla\|W\|_F^2 = 2W$; not induced
- [ ] Exercise 3: Condition number bounds precision; Tikhonov improves conditioning
- [ ] Exercise 4: Weyl's inequality; Mirsky's extension; eigenvalue instability for non-normal matrices
- [ ] Exercise 5: Nuclear norm duality; SVT is proximal operator; matrix completion via proximal gradient
- [ ] Exercise 6: Eckart-Young is exact; truncated SVD is optimal for rank-$k$ approximation in any UI norm
- [ ] Exercise 7: Power iteration converges at rate $(\sigma_2/\sigma_1)^2$; SN enforces Lipschitz=1
- [ ] Exercise 8: Factored GD implicitly minimizes nuclear norm; smaller initialization → smaller nuclear norm

## References

1. Eckart & Young (1936). The approximation of one matrix by another of lower rank. *Psychometrika*.
2. Weyl (1912). Das asymptotische Verteilungsgesetz der Eigenwerte linearer partieller Differentialgleichungen.
3. Cai, Candès & Shen (2010). A singular value thresholding algorithm for matrix completion. *SIAM J. Opt.*
4. Miyato et al. (2018). Spectral Normalization for Generative Adversarial Networks. ICLR 2018.
5. Gunasekar et al. (2017). Implicit Regularization in Matrix Factorization. NeurIPS 2017.